In [ ]:
%pip install -r requirements-offline.txt --quiet

# Worked Example — ทำครบวงจรบนราคาข้าวหอมมะลิ

**เฉลยตัวอย่าง** ของ Mini Project (เปิดเผยหลังจบช่วงทำงาน) — เดิน 6 ส่วน
บน dataset สำรอง ชุดเดียวจนจบ ให้กลุ่มที่ตันเปิดเทียบ "รูปทรง" ของแต่ละส่วน
ไม่ใช่ลอกคำตอบไปใช้กับข้อมูลตัวเอง

**ข้อมูล (citation):** ราคาข้าวเปลือกหอมมะลิ 105 รายเดือน (ม.ค. 2565 – ส.ค. 2568,
44 เดือน) — สำนักงานเศรษฐกิจการเกษตร, data.go.th (Open Data Common)

In [ ]:
import numpy as np  # noqa: E402
import pandas as pd
from pathlib import Path  # noqa: E402


def repo_root() -> Path:
    """หา repo root โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "requirements-offline.txt").exists():
            return p
    raise SystemExit(
        "ไม่พบ repo root — เปิด JupyterLab จาก repo root (uv run jupyter lab)")

DATA = repo_root() / "datasets"


## ส่วนที่ 1 — โหลดข้อมูล

แต่ละ**แถว** = ราคาเฉลี่ยรายเดือน 1 เดือน · คอลัมน์**เวลา** = `year` (พ.ศ.) +
`month` · คอลัมน์**ตัวเลขที่จะพยากรณ์** = `Value` (บาท/ตัน)

In [ ]:
rice = pd.read_csv(DATA / "rice_price_105" / "rice105_monthly.csv")
print(f"{rice.shape[0]} แถว × {rice.shape[1]} คอลัมน์")
rice.head(3)

## ส่วนที่ 2 — จัดการข้อมูลให้พร้อม

ตรวจครบชุด: ค่าว่าง · ซ้ำ · ชนิด แล้วจัดการสองขั้น —
**(1)** คอลัมน์ที่มีค่าเดียวทั้งคอลัมน์ (`_id`, `regional_name`, `commod`,
`product_name`) ไม่ช่วยตอบคำถามอะไร ตัดทิ้ง **(2)** เวลากระจายอยู่สองคอลัมน์
แถมเป็นปี **พ.ศ.** — รวมเป็นคอลัมน์ `date` เดียว (พ.ศ. − 543 = ค.ศ.)

In [ ]:
print("ค่าว่าง:", rice.isna().sum().sum(),
      "| แถวซ้ำ:", rice.duplicated().sum())
print(rice.dtypes.to_string())

rice["date"] = pd.to_datetime(
    dict(year=rice["year"] - 543, month=rice["month"], day=1))
rice = rice[["date", "year", "month", "Value"]]
print(rice["date"].min().date(), "→", rice["date"].max().date())

## ส่วนที่ 3 — สถิติที่เล่าเรื่องได้

สามประโยคที่เล่าได้จากตัวเลข: ราคาเฉลี่ยรวมทั้งช่วง ~14,100 บาท/ตัน ·
รายปี**ขึ้นเกือบทุกปี** (~900 บาท/ตันต่อปี) · เฉลี่ยรายเดือนสูงสุดตอน
ต.ค. ต่ำสุดตอน ม.ค. — มีกลิ่นของวงรอบฤดูกาล

In [ ]:
rice["Value"].describe().round(0)

In [ ]:
print(rice.groupby("year")["Value"].mean().round(0).to_string())
by_month = rice.groupby("month")["Value"].mean().round(0)
print(f"\nสูงสุด: เดือน {by_month.idxmax()} = {by_month.max():,.0f}"
      f" | ต่ำสุด: เดือน {by_month.idxmin()} = {by_month.min():,.0f}")

## ส่วนที่ 4 — กราฟที่สื่อสาร

เส้นตามเวลา (M5) เป็นกราฟหลักของงานพยากรณ์ — เส้น moving average
3 เดือน (M6) ช่วยลดเสียงรบกวนรายเดือนให้เห็นแนวโน้มชัด

In [ ]:
import matplotlib.pyplot as plt  # noqa: E402

plt.rcParams["font.family"] = [
    "DejaVu Sans", "Noto Sans Thai", "Leelawadee UI", "Tahoma",
]
rice["ma3"] = rice["Value"].rolling(3).mean()

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(rice["date"], rice["Value"], color="#9ab5c9", lw=1,
        label="ราคาจริง")
ax.plot(rice["date"], rice["ma3"], color="#457b9d", lw=2,
        label="เฉลี่ย 3 เดือน")
ax.set_xlabel("เวลา")
ax.set_ylabel("บาท/ตัน")
ax.legend()
plt.show()

In [ ]:
# กราฟเสริม: ค่าเฉลี่ยรายปี — แนวโน้มขึ้นอ่านได้ง่ายกว่าบนเส้นรายเดือน
yearly = rice.groupby("year")["Value"].mean()
fig, ax = plt.subplots(figsize=(5, 2.8))
ax.bar(yearly.index.astype(str), yearly.values, color="#457b9d")
ax.set_xlabel("ปี (พ.ศ.)")
ax.set_ylabel("บาท/ตัน (เฉลี่ยรายปี)")
plt.show()

## ส่วนที่ 5 — พยากรณ์ 3 จุดเวลาถัดไป

ข้อมูลรายเดือน → 3 จุดถัดไป = **ก.ย.–พ.ย. 2568** — ใช้ trend line
จาก `np.polyfit` บนลำดับเดือน (M6 §1)

In [ ]:
rice["seq"] = range(1, len(rice) + 1)
slope, intercept = np.polyfit(rice["seq"], rice["Value"], 1)
print(f"trend: {slope:+.0f} บาท/ตัน ต่อเดือน")

fut_seq = np.array([len(rice) + 1, len(rice) + 2, len(rice) + 3])
fut_pred = np.polyval([slope, intercept], fut_seq)
fut_date = pd.date_range(rice["date"].max(), periods=4,
                         freq="MS")[1:]

for d, v in zip(fut_date, fut_pred):
    print(f"{d.year + 543}-{d.month:02d}  พยากรณ์ {v:,.0f} บาท/ตัน")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(rice["date"], rice["Value"], color="#9ab5c9", lw=1,
        label="ราคาจริง")
ax.plot(rice["date"], np.polyval([slope, intercept], rice["seq"]),
        color="#e63946", lw=2, label="trend line")
ax.plot(fut_date, fut_pred, color="#e63946", lw=2, ls="--",
        marker="o", ms=4, label="พยากรณ์ 3 เดือน")
ax.set_xlabel("เวลา")
ax.set_ylabel("บาท/ตัน")
ax.legend()
plt.show()

## ส่วนที่ 6 — ทำไมจึงเชื่อการพยากรณ์นี้ได้

**ตรวจก่อน (ข้อ 1 ของเช็คลิสต์ M6):** ซ่อน 3 เดือนท้าย ฝึกเส้นจาก
41 เดือนแรก แล้วเทียบจริง

In [ ]:
slope_bt, intercept_bt = np.polyfit(rice["seq"][:41],
                                    rice["Value"][:41], 1)
pred_bt = np.polyval([slope_bt, intercept_bt], rice["seq"][41:])
mae = (pred_bt - rice["Value"][41:]).abs().mean()
print(f"MAE ของการซ่อนท้าย = {mae:,.0f} บาท/ตัน "
      f"(~{mae / rice['Value'].mean() * 100:.1f}% ของราคาเฉลี่ย)")
print("จริง 3 เดือนท้าย:", rice["Value"][41:].values)

**คำตอบ "ทำไมจึงเชื่อ" (อ้างเช็คลิสต์ M6):**

1. **ตรวจแล้ว** — ซ่อนท้าย 3 เดือน ทำนายผิดเฉลี่ย ~516 บาท/ตัน
   (~3.7% ของระดับราคา) รับได้ในระดับนี้
2. **ช่วงสั้นกว่าข้อมูล** — พยากรณ์ 3 เดือนจากข้อมูล 44 เดือน
3. **โลกเปลี่ยนได้** — ราคาข้าวขึ้นกับนโยบายรัฐและราคาโลกที่เปลี่ยน
   ได้ทันที นี่คือข้อที่ต้องระวังที่สุดของชุดนี้
4. **วงรอบ** — เฉลี่ยรายเดือนต.ค. สูง ม.ค. ต่ำ มีจังหวะฤดู
   แต่ 44 เดือนยังสั้นสำหรับการพยากรณ์ด้วยวงรอบเดี่ยว ๆ
5. **พูดเงื่อนไข** — *"ถ้าทุกอย่างเดินตามดังเดิม ราคาจะขึ้นแนว trend
   ~77 บาท/ตันต่อเดือน"* — และต้องยอมรับตรง ๆ ว่า 3 เดือนจริงท้าย
   (มิ.ย.–ส.ค. 2568) เริ่ม**ลดลง** ขณะ trend ยังบอกขึ้น — สัญญาณที่
   ทำให้ต้องพูดเงื่อนไขให้หนักแน่น ไม่ตอบตัดรอน

## Checklist ของตัวอย่างนี้ — ครบไหม

- [x] โหลดสำเร็จ + บอกที่มา (สำนักงานเศรษฐกิจการเกษตร, data.go.th)
- [x] จัดการข้อมูล 2 ขั้น (ตัดคอลัมน์ค่าเดียว · รวม/แปลเวลา พ.ศ.)
- [x] สถิติเล่าเป็นประโยค (เฉลี่ย · รายปี · จังหวะรายเดือน)
- [x] กราฟพร้อมป้ายแกน 2 แบบ (เส้น + แท่งรายปี)
- [x] พยากรณ์ครบ 3 จุด (ก.ย.–พ.ย. 2568) พร้อมเส้นประบนกราฟ
- [x] "ทำไมจึงเชื่อ" อ้างเช็คลิสต์ 5 ข้อ (ครบทั้ง 5)
- [x] Restart & Run All ผ่าน